# PREPARE Challenge - Phonetic Track (Wav2Vec2 Implementation)

This notebook implements a wav2vec2-based speech recognition system for phonetic transcription.

## Architecture Overview:
- **Frozen**: CNN Feature Encoder (convolutional feature extractor layers)
- **Finetuned**: Transformer Context Network (encoder layers)
- **Custom**: Linear classification head with CTC loss

## Preprocessing Pipeline:
- Resample to 16 kHz
- Convert to mono
- Normalize amplitude
- Dynamic padding within batches

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import json
import math
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW

from transformers import (
    Wav2Vec2Model,
    Wav2Vec2Config,
    Wav2Vec2FeatureExtractor,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/home/epochvipc4/Desktop/projects/speech_phonetic_track/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# =============================================================================
# DATA LOADING AND EXPLORATION
# =============================================================================

# Set paths (relative to workspace root)
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')

# --- Source 1: Original competition data ---
train_audio_dir = os.path.join(data_dir, 'audio')
train_label_path = os.path.join(data_dir, 'train_phon_transcripts.jsonl')

original_labels = pd.read_json(train_label_path, lines=True)
original_labels['file_name'] = original_labels['audio_path'].apply(os.path.basename)
original_labels['full_audio_path'] = original_labels['file_name'].apply(
    lambda f: os.path.join(train_audio_dir, f)
)
original_labels['source'] = 'original'

# --- Source 2: TalkBank additional data ---
talkbank_audio_dir = os.path.join(data_dir, 'audio_talkbank', 'audio', 'audio')
talkbank_label_path = os.path.join(data_dir, 'train_phon_transcripts_additional.jsonl')

talkbank_labels = pd.read_json(talkbank_label_path, lines=True)
talkbank_labels['file_name'] = talkbank_labels['audio_path'].apply(os.path.basename)
talkbank_labels['full_audio_path'] = talkbank_labels['file_name'].apply(
    lambda f: os.path.join(talkbank_audio_dir, f)
)
talkbank_labels['source'] = 'talkbank'

# --- Check required columns ---
required_columns = {'utterance_id', 'audio_path', 'phonetic_text'}
for name, df in [('original', original_labels), ('talkbank', talkbank_labels)]:
    if not required_columns.issubset(df.columns):
        raise ValueError(f"Missing required columns in {name}. Required: {required_columns}")

# --- Filter to files that actually exist on disk ---
original_audio_files = set(os.listdir(train_audio_dir))
talkbank_audio_files = set(os.listdir(talkbank_audio_dir))

original_matched = original_labels[original_labels['file_name'].isin(original_audio_files)]
talkbank_matched = talkbank_labels[talkbank_labels['file_name'].isin(talkbank_audio_files)]

print(f"Original:  {len(original_matched):,} / {len(original_labels):,} matched")
print(f"TalkBank:  {len(talkbank_matched):,} / {len(talkbank_labels):,} matched")

# --- Combine into single dataframe ---
audio_files_with_labels = pd.concat(
    [original_matched, talkbank_matched], ignore_index=True
)
print(f"\nCombined dataset: {len(audio_files_with_labels):,} samples")
print(f"Columns: {audio_files_with_labels.columns.tolist()}")

# Show sample phonetic transcriptions from each source
for src in ['original', 'talkbank']:
    subset = audio_files_with_labels[audio_files_with_labels['source'] == src]
    print(f"\nSample from {src} ({len(subset):,} total):")
    for i in range(min(3, len(subset))):
        row = subset.iloc[i]
        print(f"  {row['utterance_id']}: {row['phonetic_text']}")

Original:  12,043 / 12,043 matched
TalkBank:  141,024 / 141,024 matched

Combined dataset: 153,067 samples
Columns: ['utterance_id', 'child_id', 'session_id', 'audio_path', 'audio_duration_sec', 'age_bucket', 'md5_hash', 'filesize_bytes', 'phonetic_text', 'file_name', 'full_audio_path', 'source']

Sample from original (12,043 total):
  U_0004fcba47fc2b22: ʔə ʔæpɫ
  U_000727b46808376d: hjuːdə
  U_0012a1c1c3646a51: ʔɛfɹi

Sample from talkbank (141,024 total):
  U_00007df27e1aad33: əː
  U_0000891c5dd8a029: əzu
  U_00018bc22b7f1b7d: tæɹz


In [3]:
# =============================================================================
# PREPROCESSING MODULE
# =============================================================================

class AudioPreprocessor:
    """
    Audio preprocessing pipeline:
    - Resample to 16 kHz
    - Convert to mono
    - Normalize amplitude
    """
    
    def __init__(self, target_sample_rate: int = 16000):
        self.target_sample_rate = target_sample_rate
    
    def load_and_preprocess(self, audio_path: str) -> torch.Tensor:
        """
        Load audio file and apply preprocessing steps.
        
        Args:
            audio_path: Path to audio file
            
        Returns:
            Preprocessed audio tensor (1D)
        """
        # Load audio
        waveform, sample_rate = torchaudio.load(audio_path)
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # Resample to target sample rate
        if sample_rate != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(
                orig_freq=sample_rate, 
                new_freq=self.target_sample_rate
            )
            waveform = resampler(waveform)
        
        # Normalize amplitude to [-1, 1]
        waveform = waveform / (waveform.abs().max() + 1e-8)
        
        # Remove channel dimension
        waveform = waveform.squeeze(0)
        
        return waveform
    
    def apply_augmentation(
        self, 
        waveform: torch.Tensor, 
        noise_factor: float = 0.005,
        time_stretch_range: Tuple[float, float] = (0.9, 1.1)
    ) -> torch.Tensor:
        """
        Apply optional data augmentation.
        
        Args:
            waveform: Input audio tensor
            noise_factor: Standard deviation of Gaussian noise
            time_stretch_range: Range for random time stretching
            
        Returns:
            Augmented audio tensor
        """
        # # Add Gaussian noise
        # if noise_factor > 0:
        #     noise = torch.randn_like(waveform) * noise_factor
        #     waveform = waveform + noise
        
        # # Re-normalize after augmentation
        # waveform = waveform / (waveform.abs().max() + 1e-8)

        
        # left blank for now we can add stuff later if we want to do augmentation
        return waveform


# Test the preprocessor
preprocessor = AudioPreprocessor()
sample_audio_path = os.path.join(train_audio_dir, audio_files_with_labels.iloc[0]['file_name'])
sample_waveform = preprocessor.load_and_preprocess(sample_audio_path)
print(f"Sample waveform shape: {sample_waveform.shape}")
print(f"Sample rate: 16000 Hz")
print(f"Duration: {len(sample_waveform) / 16000:.2f} seconds")

Sample waveform shape: torch.Size([22960])
Sample rate: 16000 Hz
Duration: 1.44 seconds


## Vocabulary and Tokenizer Setup

Build a character-level vocabulary from the phonetic transcriptions for CTC loss.

In [4]:
# =============================================================================
# VOCABULARY AND TOKENIZER
# =============================================================================

class PhoneticVocabulary:
    """
    Build character-level vocabulary for phonetic transcription.
    
    Two separate index spaces:
    - **Label indices** (for encoding transcripts → CTC targets):
      Index 0 = BLANK, then real characters starting at 1.
      No PAD or UNK — these are NOT valid prediction targets.
    - **PAD value** = -100 (used only in collate_fn for padding labels,
      ignored by CTC loss).

    The CTC blank token is always index 0 in the output space.
    """
    
    BLANK_TOKEN = "<blank>"
    PAD_VALUE = -100  # sentinel for label padding, not a vocab entry
    
    def __init__(self):
        self.char_to_idx = {}
        self.idx_to_char = {}
        self.vocab_size = 0       # = 1 (blank) + num_characters
        
    def build_vocab(self, transcripts: List[str]) -> None:
        """
        Build vocabulary from a list of transcripts.
        Index 0 = CTC blank, then sorted characters starting at index 1.
        """
        # Collect all unique characters
        all_chars = set()
        for transcript in transcripts:
            all_chars.update(transcript)
        
        # Sort for reproducibility
        sorted_chars = sorted(list(all_chars))
        
        # Index 0 = blank, then real characters
        all_tokens = [self.BLANK_TOKEN] + sorted_chars
        self.char_to_idx = {char: idx for idx, char in enumerate(all_tokens)}
        self.idx_to_char = {idx: char for idx, char in enumerate(all_tokens)}
        self.vocab_size = len(all_tokens)
        
        print(f"Vocabulary size: {self.vocab_size}  (1 blank + {len(sorted_chars)} characters)")
        print(f"Blank index: 0")
        print(f"Sample characters: {sorted_chars[:20]}...")
        
    def encode(self, transcript: str) -> List[int]:
        """Convert transcript string to label indices (no PAD/UNK)."""
        encoded = []
        for char in transcript:
            if char in self.char_to_idx:
                encoded.append(self.char_to_idx[char])
            # Skip unknown characters entirely (don't encode them)
        return encoded
    
    def decode(self, indices: List[int], remove_blank: bool = True) -> str:
        """Convert label indices back to transcript string."""
        chars = []
        for idx in indices:
            token = self.idx_to_char.get(idx, "")
            if remove_blank and token == self.BLANK_TOKEN:
                continue
            chars.append(token)
        return ''.join(chars)
    
    @property
    def blank_idx(self) -> int:
        """CTC blank is always index 0."""
        return 0


# Build vocabulary from training data
vocab = PhoneticVocabulary()
vocab.build_vocab(audio_files_with_labels['phonetic_text'].tolist())

# Test encoding/decoding
sample_text = audio_files_with_labels.iloc[3]['phonetic_text']
encoded = vocab.encode(sample_text)
decoded = vocab.decode(encoded)
print(f"\nOriginal: {sample_text}")
print(f"Encoded:  {encoded[:20]}...")
print(f"Decoded:  {decoded}")
print(f"Match:    {sample_text == decoded}")

Vocabulary size: 53  (1 blank + 52 characters)
Blank index: 0
Sample characters: [' ', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u']...

Original: sɑnd tɔiz
Encoded:  [18, 30, 14, 4, 1, 19, 31, 9, 24]...
Decoded:  sɑnd tɔiz
Match:    True


## Validate & Clean Dataset

Scan every audio file with `torchaudio.load()`. Remove broken/undecodable files and save a cleaned manifest so we never hit runtime errors during training.

In [5]:
# =============================================================================
# VALIDATE & CLEAN DATASET — remove broken audio files
# =============================================================================
CLEAN_MANIFEST_PATH = os.path.join(data_dir, "audio_files_with_labels_clean.jsonl")

# Try to load a previously saved clean manifest to skip re-scanning
if os.path.exists(CLEAN_MANIFEST_PATH):
    audio_files_with_labels = pd.read_json(CLEAN_MANIFEST_PATH, lines=True)
    print(f"Loaded cached clean manifest: {len(audio_files_with_labels):,} samples")
else:
    broken_indices = []
    
    for idx in tqdm(range(len(audio_files_with_labels)), desc="Validating audio files"):
        fpath = audio_files_with_labels.iloc[idx]["full_audio_path"]
        try:
            waveform, sr = torchaudio.load(fpath)
            # Also reject files with 0 frames
            if waveform.shape[-1] == 0:
                broken_indices.append(idx)
        except Exception:
            broken_indices.append(idx)
    
    if broken_indices:
        broken_files = audio_files_with_labels.iloc[broken_indices][["utterance_id", "full_audio_path"]]
        print(f"\n✗ Found {len(broken_indices)} broken files:")
        for _, row in broken_files.head(10).iterrows():
            print(f"  {row['utterance_id']}: {row['full_audio_path']}")
        if len(broken_indices) > 10:
            print(f"  ... and {len(broken_indices) - 10} more")
        
        audio_files_with_labels = audio_files_with_labels.drop(index=broken_indices).reset_index(drop=True)
    else:
        print("✓ All audio files are valid")
    
    # Save cleaned manifest so we don't have to rescan next time
    audio_files_with_labels.to_json(CLEAN_MANIFEST_PATH, orient="records", lines=True)
    print(f"Saved clean manifest → {CLEAN_MANIFEST_PATH}")

print(f"Clean dataset size: {len(audio_files_with_labels):,} samples")

Validating audio files: 100%|██████████| 153067/153067 [03:06<00:00, 819.89it/s]



✗ Found 1 broken files:
  U_b8a4e8220e65219b: /home/epochvipc4/Desktop/projects/speech_phonetic_track/data/audio_talkbank/audio/audio/U_b8a4e8220e65219b.flac
Saved clean manifest → /home/epochvipc4/Desktop/projects/speech_phonetic_track/data/audio_files_with_labels_clean.jsonl
Clean dataset size: 153,066 samples


In [6]:
# =============================================================================
# DATASET MODULE
# =============================================================================
from torch.utils.data import Sampler

class PhoneticDataset(Dataset):
    """
    PyTorch Dataset for phonetic transcription.
    
    Each row must have a 'full_audio_path' column with the absolute path
    to the audio file (supports data from multiple directories).
    Padding is handled at the collation stage (see `collate_fn`).
    """
    
    def __init__(
        self,
        dataframe: pd.DataFrame,
        vocab: "PhoneticVocabulary",
        preprocessor: "AudioPreprocessor",
        augment: bool = False,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.vocab = vocab
        self.preprocessor = preprocessor
        self.augment = augment
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        audio_path = row["full_audio_path"]
        
        # Preprocess audio
        waveform = self.preprocessor.load_and_preprocess(audio_path)
        
        # Optional augmentation during training
        if self.augment:
            waveform = self.preprocessor.apply_augmentation(waveform)
        
        # Encode transcript
        label_ids = torch.tensor(self.vocab.encode(row["phonetic_text"]), dtype=torch.long)
        
        return {
            "waveform": waveform,           # (T_audio,)
            "labels": label_ids,             # (T_text,)
            "input_length": waveform.shape[0],
            "label_length": label_ids.shape[0],
        }


class BucketBatchSampler(Sampler):
    """
    Groups samples by audio duration so each batch has similar-length sequences,
    minimizing wasted padding. Buckets are shuffled each epoch for randomness.
    
    Algorithm:
      1. Sort all indices by audio_duration_sec
      2. Chunk sorted indices into buckets of size (batch_size * bucket_multiplier)
      3. Within each bucket, create batches of batch_size
      4. Shuffle the order of batches each epoch (but samples within a batch
         stay similar in length)
    """
    
    def __init__(
        self,
        durations: List[float],
        batch_size: int,
        bucket_multiplier: int = 10,
        shuffle: bool = True,
        drop_last: bool = False,
    ):
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last
        
        # Sort indices by duration
        sorted_indices = np.argsort(durations)[::-1]  # longest first
        
        # Create buckets: groups of (batch_size * bucket_multiplier) similar-length samples
        bucket_size = batch_size * bucket_multiplier
        self.batches = []
        
        for bucket_start in range(0, len(sorted_indices), bucket_size):
            bucket = sorted_indices[bucket_start : bucket_start + bucket_size].tolist()
            
            # Shuffle within each bucket for minor randomness
            if self.shuffle:
                random.shuffle(bucket)
            
            # Split bucket into batches
            for i in range(0, len(bucket), batch_size):
                batch = bucket[i : i + batch_size]
                if len(batch) == batch_size or not self.drop_last:
                    self.batches.append(batch)
    
    def __iter__(self):
        # Shuffle batch order each epoch
        batch_order = list(range(len(self.batches)))
        if self.shuffle:
            random.shuffle(batch_order)
        for idx in batch_order:
            yield self.batches[idx]
    
    def __len__(self):
        return len(self.batches)


def collate_fn(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    """
    Dynamic padding collate function.
    
    Pads waveforms and labels to the longest sequence in the batch and
    creates attention masks for padded positions.
    """
    waveforms = [item["waveform"] for item in batch]
    labels = [item["labels"] for item in batch]
    input_lengths = torch.tensor([item["input_length"] for item in batch], dtype=torch.long)
    label_lengths = torch.tensor([item["label_length"] for item in batch], dtype=torch.long)
    
    # Pad waveforms to max length in batch (dynamic padding)
    padded_waveforms = pad_sequence(waveforms, batch_first=True, padding_value=0.0)
    
    # Create attention masks: 1 for real frames, 0 for padding
    attention_mask = torch.zeros(padded_waveforms.shape, dtype=torch.long)
    for i, length in enumerate(input_lengths):
        attention_mask[i, :length] = 1
    
    # Pad labels
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return {
        "input_values": padded_waveforms,   # (B, T_max_audio)
        "attention_mask": attention_mask,     # (B, T_max_audio)
        "labels": padded_labels,             # (B, T_max_text)
        "input_lengths": input_lengths,       # (B,)
        "label_lengths": label_lengths,       # (B,)
    }


# ---------- Quick sanity check ----------
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    audio_files_with_labels, test_size=0.1, random_state=42
)
print(f"Train size: {len(train_df):,}, Val size: {len(val_df):,}")

MAX_DURATION_SEC = 5  # tune based on your VRAM
train_df = train_df[train_df['audio_duration_sec'] <= MAX_DURATION_SEC]
val_df = val_df[val_df['audio_duration_sec'] <= MAX_DURATION_SEC]

train_dataset = PhoneticDataset(train_df, vocab, preprocessor, augment=True)
val_dataset   = PhoneticDataset(val_df,   vocab, preprocessor, augment=False)

# Use BucketBatchSampler for length-grouped batching
train_sampler = BucketBatchSampler(
    durations=train_df['audio_duration_sec'].values,
    batch_size=4, shuffle=False, drop_last=False,
)
val_sampler = BucketBatchSampler(
    durations=val_df['audio_duration_sec'].values,
    batch_size=4, shuffle=False, drop_last=False,
)

train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_sampler=val_sampler,   collate_fn=collate_fn, num_workers=0)

# Peek at one batch
sample_batch = next(iter(train_loader))
for k, v in sample_batch.items():
    print(f"  {k:20s} → {v.shape}")

# Show padding efficiency
lengths = sample_batch["input_lengths"]
max_len = sample_batch["input_values"].shape[1]
efficiency = lengths.float().mean() / max_len * 100
print(f"\n  Padding efficiency: {efficiency:.1f}% (avg_len/max_len in batch)")

Train size: 137,759, Val size: 15,307
  input_values         → torch.Size([4, 80000])
  attention_mask       → torch.Size([4, 80000])
  labels               → torch.Size([4, 43])
  input_lengths        → torch.Size([4])
  label_lengths        → torch.Size([4])

  Padding efficiency: 100.0% (avg_len/max_len in batch)


## Model Architecture (wav2vec2-based)

**Frozen during training:** CNN Feature Encoder (all convolutional feature extractor layers)

**Finetuned:** Transformer Context Network (encoder layers trainable)

**Custom head:** Linear classification head → vocab-size logits → CTC loss

In [7]:
# =============================================================================
# MODEL MODULE (with LoRA)
# =============================================================================
from peft import LoraConfig, get_peft_model

class Wav2Vec2PhoneticCTC(nn.Module):
    """
    wav2vec2-based model for phonetic transcription with CTC loss.
    
    Architecture:
      - Frozen: CNN feature extractor + transformer backbone
      - LoRA adapters on attention + FFN layers
      - Custom: 2-layer MLP classification head (hidden_size → hidden_size → vocab)
    
    Output space: vocab_size = 1 (blank) + num_characters.
    PAD and UNK are NOT in the output space.
    """
    
    def __init__(
        self,
        pretrained_name: str = "facebook/wav2vec2-large-robust",
        vocab_size: int = 100,
        ctc_blank_id: int = 0,
        classifier_dropout: float = 0.1,
        lora_r: int = 16,
        lora_alpha: int = 32,
        lora_dropout: float = 0.05,
    ):
        super().__init__()
        
        self.ctc_blank_id = ctc_blank_id
        
        # Load pretrained wav2vec2 backbone
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(pretrained_name)
        hidden_size = self.wav2vec2.config.hidden_size  # 1024 for mms-300m
        
        # --- Apply LoRA to attention + FFN layers ---
        # get_peft_model freezes all base params and injects trainable LoRA adapters
        # Wav2Vec2Model has no get_input_embeddings(), so we must prevent PEFT
        # from calling enable_input_require_grads during gradient checkpointing setup.
        self.wav2vec2.supports_gradient_checkpointing = False
        self.wav2vec2.config.gradient_checkpointing = False
        # Monkey-patch: Wav2Vec2Model doesn't have token embeddings, so the
        # default enable_input_require_grads() fails. Make it a no-op.
        self.wav2vec2.enable_input_require_grads = lambda *a, **kw: None
        
        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["q_proj", "v_proj", "k_proj", "out_proj",
                            "intermediate_dense", "output_dense"],
            bias="none",
        )
        self.wav2vec2 = get_peft_model(self.wav2vec2, lora_config)
        self.wav2vec2.print_trainable_parameters()
        
        # --- 2-layer MLP classification head (no bottleneck) ---
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size),
            nn.Dropout(classifier_dropout),
            nn.Linear(hidden_size, vocab_size),
        )
        
        # Anti-blank bias: discourage blank in early training
        with torch.no_grad():
            self.head[-1].bias[ctc_blank_id] = -2.0
        print(f" Anti-blank bias at index {ctc_blank_id}")
        
        # CTC loss (blank = ctc_blank_id = 0)
        self.ctc_loss = nn.CTCLoss(blank=ctc_blank_id, zero_infinity=True)
        
        # Print full parameter summary
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Total params:     {total:,}")
        print(f"Trainable params: {trainable:,}  (LoRA r={lora_r} + head)")
        print(f"Frozen params:    {total - trainable:,}")
        print(f"Vocab size:       {vocab_size} (blank + {vocab_size - 1} chars, NO pad/unk)")
    
    def forward(
        self,
        input_values: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        outputs = self.wav2vec2(
            input_values=input_values,
            attention_mask=attention_mask,
        )
        hidden_states = outputs.last_hidden_state        # (B, T_enc, H)
        logits = self.head(hidden_states)                 # (B, T_enc, V)
        return logits
    
    def compute_ctc_loss(
        self,
        logits: torch.Tensor,
        labels: torch.Tensor,
        input_lengths: torch.Tensor,
        label_lengths: torch.Tensor,
    ) -> torch.Tensor:
        log_probs = F.log_softmax(logits, dim=-1)           # (B, T_enc, V)
        log_probs = log_probs.permute(1, 0, 2)              # (T_enc, B, V)
        
        encoder_lengths = self._get_feat_extract_output_lengths(input_lengths)
        T_enc = log_probs.size(0)
        encoder_lengths = encoder_lengths.clamp(max=T_enc)
        
        loss = self.ctc_loss(log_probs, labels, encoder_lengths, label_lengths)
        return loss
    
    def _get_feat_extract_output_lengths(self, input_lengths: torch.Tensor) -> torch.Tensor:
        def _conv_out_length(input_length, kernel_size, stride):
            return (input_length - kernel_size) // stride + 1
        
        # Access the base model's config through PEFT wrapper
        base_config = self.wav2vec2.base_model.model.config
        for kernel_size, stride in zip(
            base_config.conv_kernel, base_config.conv_stride
        ):
            input_lengths = _conv_out_length(input_lengths, kernel_size, stride)
        
        return input_lengths.to(torch.long)


# ── Thorough GPU cleanup ──
import gc

# Delete all variables that may hold GPU tensors
for _var in ['model', 'optimizer', 'scheduler', 'warmup_scheduler', 'cosine_scheduler',
             'scaler', 'backbone_params', 'head_params', 'lora_params',
             'test_logits', 'logits', 'sample_batch']:
    if _var in dir():
        try:
            exec(f'del {_var}')
        except:
            pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GiB / {torch.cuda.mem_get_info()[1] / 1024**3:.2f} GiB total")
# ---------- Instantiate model ----------
model = Wav2Vec2PhoneticCTC(
    pretrained_name="facebook/wav2vec2-base",
    vocab_size=vocab.vocab_size,
    ctc_blank_id=vocab.blank_idx,   # = 0
    classifier_dropout=0.1,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
).to(device)


# Quick forward-pass check with sample batch
sample_batch = next(iter(train_loader))
with torch.no_grad():
    test_logits = model(
        sample_batch["input_values"].to(device),
        sample_batch["attention_mask"].to(device),
    )
    print(f"\nLogits shape: {test_logits.shape}  (batch, time_enc, vocab)")

GPU memory free: 21.26 GiB / 23.46 GiB total
trainable params: 2,654,208 || all params: 97,025,920 || trainable%: 2.7356
 Anti-blank bias at index 0
Total params:     97,658,805
Trainable params: 3,287,093  (LoRA r=16 + head)
Frozen params:    94,371,712
Vocab size:       53 (blank + 52 chars, NO pad/unk)

Logits shape: torch.Size([4, 249, 53])  (batch, time_enc, vocab)


## Training Setup

- **Loss function:** CTC loss
- **Optimizer:** AdamW
- **LR scheduler:** Linear warmup + decay
- **Gradient clipping:** max norm = 1.0
- **Includes:** Validation step every epoch

In [8]:
# =============================================================================
# TRAINING LOOP
# =============================================================================
from torch.amp import autocast, GradScaler
@dataclass
class TrainingConfig:
    """All hyperparameters in one place."""
    epochs: int = 50
    batch_size: int = 8
    lora_lr: float = 2e-5          # Lower LR for LoRA adapters (fine-tuning pretrained weights)
    head_lr: float = 5e-4          # Higher LR for randomly-initialized head
    weight_decay: float = 0.01
    warmup_ratio: float = 0.15
    max_grad_norm: float = 1.0
    num_workers: int = 2
    log_every: int = 50
    save_dir: str = "checkpoints"


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    config: TrainingConfig,
    epoch: int,
    scaler: GradScaler,
) -> float:
    """Train for one epoch. Returns average loss."""
    model.train()
    running_loss = 0.0
    total_steps = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} [train]", leave=False)
    for step, batch in enumerate(pbar):
        input_values  = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        input_lengths  = batch["input_lengths"].to(device)
        label_lengths  = batch["label_lengths"].to(device)
        
        # Forward (autocast for backbone + head, CTC loss in float32)
        with autocast('cuda'):
            logits = model(input_values, attention_mask)
            loss = model.compute_ctc_loss(logits, labels, input_lengths, label_lengths)
        
        # Backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        
        # Gradient clipping
        clip_grad_norm_(model.parameters(), config.max_grad_norm)
        
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        running_loss += loss.item()
        total_steps += 1
        
        if (step + 1) % config.log_every == 0:
            avg = running_loss / total_steps
            pbar.set_postfix(loss=f"{avg:.4f}")
    
    return running_loss / max(total_steps, 1)


@torch.no_grad()
def validate(model: nn.Module, loader: DataLoader) -> float:
    """Run validation. Returns average CTC loss."""
    model.eval()
    running_loss = 0.0
    total_steps = 0
    
    for batch in tqdm(loader, desc="  [val]", leave=False):
        input_values   = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        input_lengths  = batch["input_lengths"].to(device)
        label_lengths  = batch["label_lengths"].to(device)
        with autocast('cuda'):
            logits = model(input_values, attention_mask)
            loss = model.compute_ctc_loss(logits, labels, input_lengths, label_lengths)
        
        running_loss += loss.item()
        total_steps += 1
    
    return running_loss / max(total_steps, 1)

# beam search later add``
def greedy_ctc_decode(logits: torch.Tensor, vocab: "PhoneticVocabulary") -> List[str]:
    """
    Greedy CTC decoding: argmax → collapse repeats → remove blank.
    
    Since PAD and UNK are no longer in the output vocab, the only
    special token to filter is BLANK (index 0).
    """
    blank_id = vocab.blank_idx  # = 0
    preds = logits.argmax(dim=-1)  # (B, T)
    
    decoded = []
    for seq in preds:
        tokens = []
        prev = -1
        for t in seq:
            t = t.item()
            if t != blank_id and t != prev:
                tokens.append(t)
            prev = t
        decoded.append(vocab.decode(tokens))
    return decoded


print("Training utilities defined ✓")

Training utilities defined ✓


## Run Training

Execute the full training pipeline: build data loaders, optimizer, scheduler, and train with validation.

In [9]:
# =============================================================================
# RUN TRAINING
# =============================================================================
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, SequentialLR, LinearLR, CosineAnnealingLR
from torch.amp import GradScaler, autocast

config = TrainingConfig(
    epochs=20,
    batch_size=32,
    lora_lr=2e-5,             # LoRA adapters: gentler LR for large model
    head_lr=5e-4,             # Classification head: higher LR (randomly initialized)
    weight_decay=0.01,
    warmup_ratio=0.15,
    max_grad_norm=1.0,
    num_workers=2,
    log_every=50,
    save_dir="checkpoints",
)

# --- Data loaders with length-bucketed batching ---
train_sampler = BucketBatchSampler(
    durations=train_df['audio_duration_sec'].values,
    batch_size=config.batch_size,
    shuffle=True,
    drop_last=False,
)
val_sampler = BucketBatchSampler(
    durations=val_df['audio_duration_sec'].values,
    batch_size=config.batch_size,
    shuffle=False,
    drop_last=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    collate_fn=collate_fn,
    num_workers=config.num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_sampler=val_sampler,
    collate_fn=collate_fn,
    num_workers=config.num_workers,
    pin_memory=True,
)

# --- Differential LR: LoRA adapters vs classification head ---
# Only trainable params are included (backbone is frozen by LoRA/PEFT)
lora_params = [p for n, p in model.named_parameters()
               if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters()
               if p.requires_grad and "head" in n]

print(f"LoRA adapter params: {sum(p.numel() for p in lora_params):,}")
print(f"Head params:         {sum(p.numel() for p in head_params):,}")

optimizer = AdamW([
    {"params": lora_params, "lr": config.lora_lr},
    {"params": head_params, "lr": config.head_lr},
], weight_decay=config.weight_decay)

# --- Cosine annealing scheduler (doesn't decay to 0) ---
total_steps = len(train_loader) * config.epochs

scheduler = CosineAnnealingLR(optimizer, T_max=total_steps , eta_min=1e-6)
scaler = GradScaler()

print(f"Total training steps: {total_steps}")
print(f"Train batches/epoch:  {len(train_loader)}")
print(f"Val batches/epoch:    {len(val_loader)}")

# --- Training loop ---
train_losses = []
val_losses = []
best_val_loss = float("inf")

os.makedirs(config.save_dir, exist_ok=True)

for epoch in range(config.epochs):
    # Train
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, config, epoch, scaler)
    train_losses.append(train_loss)
    
    # Validate
    val_loss = validate(model, val_loader)
    val_losses.append(val_loss)

    # Print every 5 epochs or first 5
    if epoch < 5 or (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{config.epochs}  │  train_loss={train_loss:.4f}  │  val_loss={val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_path = os.path.join(config.save_dir, "best_model.pt")
        torch.save(model.state_dict(), save_path)
        if epoch < 5 or (epoch + 1) % 5 == 0:
            print(f"  ✓ Saved best model (val_loss={val_loss:.4f})")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

LoRA adapter params: 2,654,208
Head params:         632,885
Total training steps: 79080
Train batches/epoch:  3954
Val batches/epoch:    440


Epoch 1/20  │  train_loss=2.3336  │  val_loss=1.4939
  ✓ Saved best model (val_loss=1.4939)


Epoch 2/20  │  train_loss=1.7366  │  val_loss=1.3723
  ✓ Saved best model (val_loss=1.3723)


Epoch 3/20  │  train_loss=1.6192  │  val_loss=1.2770
  ✓ Saved best model (val_loss=1.2770)


Epoch 4/20  │  train_loss=1.5560  │  val_loss=1.2385
  ✓ Saved best model (val_loss=1.2385)


Epoch 5/20  │  train_loss=1.5050  │  val_loss=1.1963
  ✓ Saved best model (val_loss=1.1963)


Epoch 10/20  │  train_loss=1.3609  │  val_loss=1.0979
  ✓ Saved best model (val_loss=1.0979)


RuntimeError: Parent directory checkpoints does not exist.

## Evaluation & Visualization

Plot training curves and inspect sample predictions via greedy CTC decoding.

In [ ]:
# =============================================================================
# EVALUATION & VISUALIZATION
# =============================================================================

# --- Plot training curves ---
fig, ax = plt.subplots(figsize=(10, 5))
epochs_range = range(2, len(train_losses) + 1)
ax.plot(epochs_range, train_losses[1:], label="Train Loss", marker="o")
ax.plot(epochs_range, val_losses[1:], label="Val Loss", marker="s")
ax.set_xlabel("Epoch")
ax.set_ylabel("CTC Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Load best model and show sample predictions ---
best_path = os.path.join(config.save_dir, "best_model.pt")
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
    print("Loaded best model checkpoint\n")

model.eval()
sample_batch = next(iter(val_loader))
with torch.no_grad():
    logits = model(
        sample_batch["input_values"].to(device),
        sample_batch["attention_mask"].to(device),
    )

predictions = greedy_ctc_decode(logits, vocab)

print("=" * 60)
print("SAMPLE PREDICTIONS (greedy CTC decode)")
print("=" * 60)
for i in range(min(len(predictions), 6)):
    # Ground truth: decode label ids (ignore -100 padding)
    gt_ids = sample_batch["labels"][i]
    gt_ids = gt_ids[gt_ids != -100].tolist()
    gt_text = vocab.decode(gt_ids)
    
    print(f"\n  Ground truth: {gt_text}")
    print(f"  Prediction:   {predictions[i]}")

## Competition Metric (IPA-CER / PER) & Confusion Matrix

Evaluate model performance using:
1. **IPA-CER (PER)** — the official competition metric from `src/utils/score.py`. Character Error Rate on normalized IPA transcriptions.
2. **Confusion Matrix** — character-level substitution analysis to identify which phonemes the model confuses most.

In [ ]:
import sys, os, torch, numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from jiwer import cer as jiwer_cer

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
from utils.score import score_ipa_cer, normalize_ipa

# 1. Collect predictions (Optimized Inference Loop)
model.eval()
all_preds, all_refs = [], []
with torch.no_grad():
    for b in tqdm(val_loader, desc="Collecting predictions"):
        logits = model(b["input_values"].to(device), b["attention_mask"].to(device))
        all_preds.extend(greedy_ctc_decode(logits, vocab))
        # One-liner for decoding ground-truth labels (ignoring padding -100)
        all_refs.extend([vocab.decode(lbl[lbl != -100].tolist()) for lbl in b["labels"]])

# 2. Compute Metrics (Overall & Per-Sample)
ipa_cer = score_ipa_cer(actual=all_refs, predicted=all_preds)

# One-liner for per-sample CER
per_sample = np.array([
    jiwer_cer(normalize_ipa(r), normalize_ipa(p)) if normalize_ipa(r) else float(bool(normalize_ipa(p)))
    for r, p in zip(all_refs, all_preds)
])

# 3. Plot Distribution (Dark Mode + Seaborn)
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("#FFFFFF")


# Histplot automatically handles binning and edges cleanly
sns.histplot(per_sample, bins=30, color="#bb86fc", edgecolor="#121212", ax=ax)
ax.axvline(per_sample.mean(), color="#cf6679", ls="--", lw=2, label=f"Mean={per_sample.mean():.3f}")
ax.axvline(np.median(per_sample), color="#03dac6", ls="--", lw=2, label=f"Median={np.median(per_sample):.3f}")

# Compact label setting
ax.set(xlabel="IPA-CER per sample", ylabel="Count", title=f"IPA-CER Distribution (Overall: {ipa_cer:.3f})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter
import Levenshtein
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Align sequences and collect ALL unique characters
subs = Counter()
all_chars = set()

for ref, pred in zip(all_refs, all_preds):
    r, p = normalize_ipa(ref), normalize_ipa(pred)
    
    # Collect every character encountered
    all_chars.update(r)
    all_chars.update(p)
    
    # Count substitutions
    for op, i, j in Levenshtein.editops(r, p):
        if op == 'replace': 
            subs[(r[i], p[j])] += 1

chars = sorted(all_chars)

# 2. Build the full confusion matrix dataframe
df = pd.DataFrame(0, index=chars, columns=chars)
for (r, p), count in subs.items():
    df.at[r, p] += count

# 3. Plot with upgraded aesthetics
sns.set_theme(style="white") # Clean base theme
fig, ax = plt.subplots(figsize=(max(12, len(chars) * 0.4), max(10, len(chars) * 0.35)))

# Apply a nice soft background color to the figure and axes
bg_color = "#CBC1A4" 
fig.patch.set_facecolor(bg_color)
ax.set_facecolor(bg_color)

# Draw the heatmap (mask=df==0 lets the background color show through empty cells)
sns.heatmap(df, annot=True, fmt="d", cmap="YlOrRd", 
            mask=df==0, cbar_kws={'label': 'Count'}, ax=ax)

plt.xlabel("Predicted character", fontweight='bold')
plt.ylabel("Reference character", fontweight='bold')
plt.title("Full Phoneme Confusion Matrix", pad=15, fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()